# STO Ti-L dipole density matrix with pyms_plus

This standalone example reads `SR_EELS/data/structures/STO3bc.cif` and computes a projected Ti density matrix. It uses only pyms_plus and packages in the py_multislice_plus environment. The three analytic dipole transition potentials follow the approach in the first cells of `Simulation_demo_STO.ipynb`; they are a simplified model, not quantitative atomic potentials. The matrix scale is arbitrary.

Run from this directory or another directory inside the checkout. Increase `thickness`, `gridshape` and `nT` for a larger simulation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import dask
import dask.base as dask_base

checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "SR_EELS/data/structures/STO3bc.cif").is_file()
     and (p / "simulation/inelastic/TEM_related/py_multislice_plus/pyms_plus").is_dir()),
    None,
)
if checkout is None:
    raise FileNotFoundError("Run inside the checkout containing SR_EELS and py_multislice_plus")
package_root = checkout / "simulation/inelastic/TEM_related/py_multislice_plus"
sys.path.insert(0, str(package_root))
from pyms_plus.density_matrix import calculate_dm
from pyms_plus.density_matrix_setup import StructureASE, load_structure
from pyms_plus.Probe import wavev

# abTEM only needs Dask's local scheduler here. Avoid optional distributed
# discovery, which can fail on Windows with a broken system SSL store.
dask.config.set(scheduler='single-threaded')
dask_base._distributed_available = lambda: False

structure_file = checkout / "SR_EELS/data/structures/STO3bc.cif"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| structure:", structure_file)

## Specimen and dipole transition potentials

The CIF's third lattice vector is the beam direction. We repeat the short lateral direction three times and keep atoms within the requested thickness. The density-matrix routine expects the structure to contain only target atoms in that slab. `axis=1` keeps coherence along the repeated lateral direction.

In [ ]:
eV = 200_000.0               # accelerating energy, eV
energy_loss = 456.0          # representative Ti L-edge loss, eV
thickness = 8.0              # angstrom
slice_thickness = 2.0        # angstrom
gridshape = (64, 48)
nT = 1                      # increase for frozen-phonon averaging

atoms = load_structure(structure_file, repeat=(1, 3, 1)).atoms
atoms = atoms[(atoms.positions[:, 2] >= 0) & (atoms.positions[:, 2] < thickness)]
cell = atoms.cell.array.copy()
cell[2, 2] = thickness
atoms.set_cell(cell, scale_atoms=False)
crystal = StructureASE(atoms)
assert len(crystal.get_target_positions(22)) > 0, "No Ti sites in the slab"
rsize = crystal.unitcell[:2]
print(f"{len(atoms)} atoms, {len(crystal.get_target_positions(22))} Ti sites; lateral size {rsize} Å")

# A simple three-channel dipole model in reciprocal space.
qy = np.fft.fftfreq(gridshape[0], d=rsize[0] / gridshape[0])[:, None]
qx = np.fft.fftfreq(gridshape[1], d=rsize[1] / gridshape[1])[None, :]
qz = wavev(eV) - wavev(eV - energy_loss)
theta = np.deg2rad(7.0)
denominator = qy**2 + qx**2 + qz**2
Hn0 = np.stack([
    np.fft.ifft2((qy * np.cos(theta) - qz * np.sin(theta)) / denominator),
    np.fft.ifft2(np.broadcast_to(qx, gridshape) / denominator),
    np.fft.ifft2((qy * np.sin(theta) + qz * np.cos(theta)) / denominator),
]).astype(np.complex64)
print("Transition potentials:", Hn0.shape)

## Propagate and form the density matrix

The three dipole channels add incoherently. `n`, `ell` and `epsilon` label the nominal Ti 2p transition; the supplied `Hn0` defines the actual transition model.

In [ ]:
rho, rho_channels = calculate_dm(
    crystal, gridshape, eV, app=None, thicknesses=thickness,
    Ztarget=22, n=2, ell=1, epsilon=1, Hn0=Hn0,
    slice_thickness=slice_thickness, device_type=device,
    nT=nT, axis=1, return_components=True, showProgress=True,
)
assert rho.shape == (gridshape[1], gridshape[1])
assert np.all(np.isfinite(rho))
assert np.allclose(rho, rho_channels.sum(axis=0), rtol=2e-5, atol=1e-8)
assert np.allclose(rho, rho.conj().T, rtol=1e-4, atol=1e-8)
print("Density matrix:", rho.shape, "| trace:", float(np.trace(rho).real))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
extent = [0, rsize[1], rsize[1], 0]
im = axes[0].imshow(rho.real, origin="upper", extent=extent, cmap="RdBu_r")
axes[0].set(title="Re density matrix", xlabel="y' (Å)", ylabel="y (Å)")
fig.colorbar(im, ax=axes[0], label="arbitrary units")
y = (np.arange(rho.shape[0]) + 0.5) * rsize[1] / rho.shape[0]
axes[1].plot(y, np.diag(rho).real)
axes[1].set(title="Diagonal intensity", xlabel="y (Å)", ylabel="arbitrary units")
fig.tight_layout()
plt.show()